In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [22]:
from langchain.tools import tool

@tool
def suma(a: float, b: float) -> float:
    """Agregar dos sumeros"""
    return a + b

@tool
def multi(a: float, b: float) -> float:
    """Multiplica dos numeros"""
    return a * b

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

In [23]:
from langchain.agents import create_agent

# create subagents

subagent_suma = create_agent(
    model='gpt-5-nano',
    tools=[suma]
)

subagent_multi = create_agent(
    model='gpt-5-nano',
    tools=[multi]
)

## Calling subagents

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

In [24]:
from langchain.messages import HumanMessage

@tool
def call_subagent_suma(x: float) -> float:
    """Llama a agente sumar"""
    response = subagent_suma.invoke({"messages": [HumanMessage(content=f"Sumar {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_multi(x: float) -> float:
    """Llama a agente multicar"""
    response = subagent_multi.invoke({"messages": [HumanMessage(content=f"Multiplicar {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent_operation = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_suma, call_subagent_multi],
    system_prompt="Usa subagentes.")

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='3f84520a-eff4-4c7c-a1d8-613b3aaaa627'),
              AIMessage(content='Approximately 21.354156. (Since √456 is irrational, this is a decimal approximation.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2206, 'prompt_tokens': 202, 'total_tokens': 2408, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2176, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Ef445Qamq6CtjlXq8uCQQGvfuZjRn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1924b-6fd9-7081-86b5-df9ed22d48cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 2206, 'tot

In [27]:
question2 = "Multiplicar 5 por 3"

response = main_agent_operation.invoke({"messages": [HumanMessage(content=question2)]})

In [28]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Multiplicar 5 por 3', additional_kwargs={}, response_metadata={}, id='acae153a-14d9-42ee-88c1-19fa29372090'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1307, 'prompt_tokens': 166, 'total_tokens': 1473, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1280, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Ef5CPLF1wf449oxGGE9dzuzSdVQVi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1928d-fc2e-7e10-bc7f-2d5df2671d34-0', tool_calls=[{'name': 'call_subagent_multi', 'args': {'x': 3}, 'id': 'call_NIll6E6r76WqxZhXB65NyuFv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_t